# Spotify Songs 2023 - Exploratory Data Analysis

## Research Question
Which audio features are most strongly associated with Spotify streaming success in 2023?

### Step 1: Import Libraries and Load the Dataset

In this step, I import the necessary Python libraries for data cleaning, model training, and model evaluation. Since the goal is to predict the popularity of songs based on available features, I will use XGBoost Regressor as a tree-based boosting model. XGBoost is useful because it can capture non-linear relationships and feature interactions better than a simple linear regression model.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

df = pd.read_csv("spotify-2023.csv", encoding="latin1")

df.head()

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6


### Step 2: Data Inspection and Cleaning

In this step, I examine the structure of the dataset and check for potential data quality issues. This includes identifying missing values, understanding data types, and preparing the target variable.

One key issue in this dataset is that the "streams" column is stored as a string with commas, which prevents it from being used in numerical modeling. Therefore, I convert it into a numeric format.

Additionally, since the distribution of streams is highly right-skewed, I apply a log transformation to create a new variable called "log_streams". This helps stabilize variance and improves model performance.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   track_name            953 non-null    str  
 1   artist(s)_name        953 non-null    str  
 2   artist_count          953 non-null    int64
 3   released_year         953 non-null    int64
 4   released_month        953 non-null    int64
 5   released_day          953 non-null    int64
 6   in_spotify_playlists  953 non-null    int64
 7   in_spotify_charts     953 non-null    int64
 8   streams               953 non-null    str  
 9   in_apple_playlists    953 non-null    int64
 10  in_apple_charts       953 non-null    int64
 11  in_deezer_playlists   953 non-null    str  
 12  in_deezer_charts      953 non-null    int64
 13  in_shazam_charts      903 non-null    str  
 14  bpm                   953 non-null    int64
 15  key                   858 non-null    str  
 16  mode               

In [4]:
df.isnull().sum()

track_name               0
artist(s)_name           0
artist_count             0
released_year            0
released_month           0
released_day             0
in_spotify_playlists     0
in_spotify_charts        0
streams                  0
in_apple_playlists       0
in_apple_charts          0
in_deezer_playlists      0
in_deezer_charts         0
in_shazam_charts        50
bpm                      0
key                     95
mode                     0
danceability_%           0
valence_%                0
energy_%                 0
acousticness_%           0
instrumentalness_%       0
liveness_%               0
speechiness_%            0
dtype: int64

In [5]:
df['streams'] = pd.to_numeric(df['streams'], errors='coerce')

df['streams'].head()
df['streams'].dtype

dtype('float64')

In [6]:
df['log_streams'] = np.log1p(df['streams'])

df[['streams', 'log_streams']].head()

,streams,log_streams
0,141381703.0,18.766974
1,133716286.0,18.711231
2,140003974.0,18.757181
3,800840817.0,20.501173
4,303236322.0,19.530023


### Step 3: Feature Selection

In this step, I select relevant features to use in the model. The goal is to identify which variables may help explain the variation in song streaming performance.

I start by selecting a set of audio-related features such as danceability, energy, and acousticness. These features describe the characteristics of the songs and are commonly used in music analysis.

The target variable is "log_streams", which represents the transformed streaming count.

In [7]:
audio_features = [
    'danceability_%',
    'energy_%',
    'valence_%',
    'acousticness_%',
    'instrumentalness_%',
    'liveness_%',
    'speechiness_%'
]

X = df[audio_features]
y = df['log_streams']

### Step 4: Train-Test Split

In this step, I split the dataset into training and testing sets. The purpose is to evaluate the model's ability to generalize to unseen data.

The training set is used to train the model, while the testing set is used to evaluate its performance. This helps prevent overfitting and ensures that the model is not simply memorizing the data.

I use an 80-20 split and set a random_state to ensure reproducibility.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

### Step 4.5: Check and Handle Missing Values

Before training the XGBoost model, I check whether the selected features or target variable contain missing values. Missing values can cause model training errors or reduce model reliability.

Since the selected audio features and the target variable are essential for this model, I remove rows that contain missing values in these columns before splitting the data.

In [9]:
df[audio_features + ['log_streams']].isnull().sum()

danceability_%        0
energy_%              0
valence_%             0
acousticness_%        0
instrumentalness_%    0
liveness_%            0
speechiness_%         0
log_streams           1
dtype: int64

In [10]:
model_df = df[audio_features + ['log_streams']].dropna()

X = model_df[audio_features]
y = model_df['log_streams']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

### Step 5: Train XGBoost Model

In this step, I train an XGBoost regression model using the selected features. XGBoost is an ensemble learning method that builds multiple decision trees sequentially and improves performance by reducing errors from previous trees.

This model is particularly effective for capturing complex, non-linear relationships between features and the target variable.

In [11]:
model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_met

### Step 6: Model Evaluation

In this step, I evaluate the performance of the XGBoost model using the test dataset. I generate predictions and compare them with the actual values.

To measure the model performance, I use three metrics:
- Mean Absolute Error (MAE): measures the average absolute difference between predicted and actual values.
- Root Mean Squared Error (RMSE): penalizes larger errors more heavily.
- R-squared (R²): measures how much variance in the target variable is explained by the model.

These metrics help assess how well the model generalizes to unseen data.

In [12]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 0.8247475473534752
RMSE: 1.0151053493762026
R²: -0.002828716169715806


### Step 7: Exploring Alternative Features

Since audio features show very low explanatory power, I explore other types of features that may better explain streaming performance.

These include variables related to artist presence, release information, and platform exposure. These features are more likely to capture external factors influencing song popularity.

In [13]:
other_features = [
    'artist_count',
    'released_year',
    'released_month',
    'released_day'
]

model_df = df[other_features + ['log_streams']].dropna()

X = model_df[other_features]
y = model_df['log_streams']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R²:", r2_score(y_test, y_pred))

R²: 0.48773026240251804


### Step 8 : Exposure Features Analysis

In this step, I analyze the impact of exposure-related features on streaming performance. These features include the number of times a song appears in playlists and charts across different platforms such as Spotify, Apple Music, and Deezer.

Before training the model, I convert all exposure features into numeric format to ensure compatibility with the XGBoost model. Any non-numeric values are coerced into missing values and removed.

I then train an XGBoost regression model using these exposure features and evaluate its performance using the R² metric.

The results show a significantly higher R² compared to both audio features and basic metadata features. This indicates that exposure-related variables have much stronger explanatory power for streaming performance.

This suggests that a song’s popularity is largely driven by platform visibility, playlist inclusion, and chart presence, rather than its intrinsic audio characteristics.

In [18]:
exposure_features = [
    'in_spotify_playlists',
    'in_spotify_charts',
    'in_apple_playlists',
    'in_apple_charts',
    'in_deezer_playlists',
    'in_deezer_charts'
]
for col in exposure_features:
    df[col] = pd.to_numeric(df[col], errors='coerce')
model_df = df[exposure_features + ['log_streams']].dropna()

X = model_df[exposure_features]
y = model_df['log_streams']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R²:", r2_score(y_test, y_pred))

R²: 0.343760892767933


### Step 9: Feature Importance Analysis

In this step, I analyze the importance of individual features within each feature group. Feature importance indicates how much each variable contributes to the model’s predictions.

By examining feature importance, I can identify which specific factors are most influential in explaining streaming performance.

This analysis is conducted separately for different feature groups to better understand their relative contributions.

In [20]:
model_df_other = df[other_features + ['log_streams']].dropna()

X_other = model_df_other[other_features]
y_other = model_df_other['log_streams']


model_other = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

model_other.fit(X_other, y_other)


other_importance = pd.DataFrame({
    'feature': other_features,
    'importance': model_other.feature_importances_
}).sort_values(by='importance', ascending=False)

other_importance

,feature,importance
1,released_year,0.640925
2,released_month,0.187651
3,released_day,0.096362
0,artist_count,0.075062


In [21]:
for col in exposure_features:
    df[col] = pd.to_numeric(df[col], errors='coerce')

model_df_exposure = df[exposure_features + ['log_streams']].dropna()

X_exposure = model_df_exposure[exposure_features]
y_exposure = model_df_exposure['log_streams']

model_exposure = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

model_exposure.fit(X_exposure, y_exposure)

exposure_importance = pd.DataFrame({
    'feature': exposure_features,
    'importance': model_exposure.feature_importances_
}).sort_values(by='importance', ascending=False)

exposure_importance

,feature,importance
0,in_spotify_playlists,0.598942
1,in_spotify_charts,0.105464
5,in_deezer_charts,0.098745
4,in_deezer_playlists,0.092077
2,in_apple_playlists,0.052864
3,in_apple_charts,0.051908


The feature importance results clearly show that playlist exposure on Spotify is the dominant factor influencing streaming performance.

Specifically, the variable "in_spotify_playlists" accounts for nearly 60% of the total importance, far exceeding all other features. This indicates that being included in Spotify playlists significantly increases a song’s visibility and likelihood of being streamed.

In contrast, chart-related features and exposure on other platforms such as Apple Music and Deezer contribute much less to the model. While they still have some influence, their impact is relatively minor compared to Spotify playlist placement.

This suggests that Spotify plays a central role in shaping music consumption behavior, and that playlist curation is a key mechanism through which songs gain popularity.

Overall, the results reinforce the idea that streaming success is primarily driven by platform exposure, especially through Spotify playlists, rather than by the intrinsic characteristics of the music itself.